# 02c — TUM EV UDS Extraction

This notebook demonstrates the memory-safe extraction of the TUM dataset.

## ⚠️ WARNING
The TUM dataset contains ~98 million raw rows across 7 Parquet files.
Loading the entire dataset into pandas will likely exhaust RAM on a 16GB machine.

This notebook safely inspects one row group using PyArrow to verify extraction logic.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import pyarrow.parquet as pq
import pyarrow as pa
import pandas as pd

from scripts.tum_extractor import REQUIRED_IDS, get_memory_usage_mb

## 1. Inspect Single Row Group

Instead of loading an entire file, we load just the first Row Group of `CUP1.parquet` to verify the schema and values.

In [ ]:
sample_file = '../dataset/electric-vehicle-uds-dataset-main/data/uds_data/CUP1.parquet'

if os.path.exists(sample_file):
    pf = pq.ParquetFile(sample_file)
    print(f"File: CUP1.parquet")
    print(f"Total Rows: {pf.metadata.num_rows:,}")
    print(f"Total Row Groups: {pf.num_row_groups}")
    print(f"Columns: {pf.metadata.schema.names}\n")
    
    print("--- Processing First Row Group ---")
    ram_before = get_memory_usage_mb()
    
    # Read just RG 0
    rg0_table = pf.read_row_group(0)
    print(f"Row Group 0 rows: {rg0_table.num_rows:,}")
    
    # Filter for REQUIRED_IDS
    filtered_table = rg0_table.filter(
        pa.compute.is_in(rg0_table['value_id'], value_set=pa.array(list(REQUIRED_IDS.keys())))
    )
    print(f"Filtered rows (required signals only): {filtered_table.num_rows:,}")
    
    ram_after = get_memory_usage_mb()
    print(f"RAM used: {ram_after - ram_before:.2f} MB (Total: {ram_after:.2f} MB)")
    
    # Show sample
    df_sample = filtered_table.to_pandas().head(10)
    df_sample['signal_name'] = df_sample['value_id'].map(REQUIRED_IDS)
    display(df_sample)
else:
    print("Dataset not found at expected location.")

## 2. Full Extraction Execution

To run the full extraction on all 7 vehicles safely, it is recommended to run the script from the terminal to avoid Jupyter notebook memory overhead overheads.

```bash
python scripts/tum_extractor.py
```

*Uncomment the cell below if you wish to run it directly in this notebook (monitor your RAM!).*

In [ ]:
# from scripts.tum_extractor import main
# main()

## 3. Verify Extracted Output

Once extraction is complete, we can safely inspect the summarized statistics.

In [ ]:
stats_file = '../data/interim/tum/tum_signal_statistics.csv'
if os.path.exists(stats_file):
    stats_df = pd.read_csv(stats_file)
    display(stats_df)
else:
    print("Extraction hasn't been run yet.")